# Train a Spam Classifier: Student Notebook

This notebook walks through what `train.py` does, one step at a time. You will train a model, learn how to measure it properly, and save it so other people (and future you) can trust it.

| Section | What to do |
|---|---|
| **Part A · Starter** | Read the explanations, then fill in every `# TODO`. Run each cell as you go. |
| **Part B · Solution** | Working code and answers for every step. Try Part A first, then compare. |

**Before you start:** open this notebook from the folder that contains `make_data.py`, and run `python make_data.py` once, so the `data/` files exist.

> The diagrams in this notebook are Mermaid charts. They render in JupyterLab 4.1+, Notebook 7.1+ and VS Code. In older versions you will see the diagram source as text instead.

### The big picture

Every ML model goes through the same loop. This notebook covers steps 1 to 6. The rest of the lab (pipeline, rollback, drift) covers the dashed arrow: what to do when new data arrives.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    A["1 Load data"] --> B["2 Split"]
    B --> C["3 Train"]
    C --> D["4 Evaluate"]
    D --> E["5 Save"]
    E --> F["6 Predict"]
    F -. "new data" .-> A
```

### Key words

| Word | Meaning in this notebook |
|---|---|
| **ham** | A normal message you want to read |
| **spam** | An unwanted message |
| **feature** | A number we measure about each message. The model only sees features, never the text. |
| **label** | The right answer for a message: `ham` or `spam` |
| **training** | Showing the model many messages *with* their labels so it learns a rule |
| **prediction** | The model's guess for a message it has not seen |

Our model only looks at **two features**:

| Feature | Meaning | `"WINNER!! Claim your free prize now!!!"` |
|---|---|---|
| `length` | number of characters | 37 |
| `punct` | number of punctuation marks | 5 |

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    T["Message text"] --> X["extract_features"]
    X --> L["length = 37"]
    X --> P["punct = 5"]
    L --> M["Model"]
    P --> M
    M --> Y["ham or spam"]
```

The data files you will use:

| File | What it is |
|---|---|
| `data/sms_real_1k.csv` | Real messages: 85% ham, 15% spam |
| `data/balanced.csv` | 1,000 ham + 1,000 spam |
| `data/test_set.csv` | A **fixed** test set we always use to compare models |

---
# Part A · Starter

Replace each `None` or `...` marked `# TODO` with working code. Hints are in the comments.

## A0 · Setup

Run this cell as is. It imports the libraries and checks that the data files exist.

In [ ]:
import hashlib
import json
import os
import string
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split

FEATURES = ["length", "punct"]
LABEL = "label"

for f in ["data/sms_real_1k.csv", "data/balanced.csv", "data/test_set.csv"]:
    assert os.path.exists(f), f"{f} is missing. Run `python make_data.py` in this notebook's folder first."
print("Data files found.")

## A1 · Load and explore the real data

Always look at your data before training. Two questions matter most:

1. **How many of each label?** If one label is rare, the model may learn to ignore it.
2. **Do the features differ between labels?** If spam and ham look the same on our features, no model can tell them apart.

The real data looks like this:

```mermaid
pie showData title sms_real_1k.csv, 1,000 messages
    "ham" : 850
    "spam" : 150
```

A dataset like this, where one label is much more common, is called **imbalanced**.

In [ ]:
# TODO: load data/sms_real_1k.csv into a DataFrame
# Hint: pd.read_csv(...)
real = None

# TODO: show the first 5 rows
# Hint: real.head()
...

In [ ]:
# TODO: print the share of each label
# Hint: real["label"].value_counts(normalize=True)
...

# TODO: show the average length and punct for ham vs spam
# Hint: real.groupby("label")[FEATURES].mean()
...

**Q1.** What share of the messages is spam? Do spam messages look different from ham on these two features?

## A2 · Split into train and test

If we test a model on the same messages it learned from, it can just remember the answers. That tells us nothing about new messages. So we hide some data away before training:

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    D["sms_real_1k.csv<br/>1,000 rows"] --> S{"split"}
    S -- "75%" --> TR["train_df<br/>750 rows<br/>model learns"]
    S -- "25%" --> TE["test_df<br/>250 rows<br/>kept hidden"]
```

Two settings matter:

| Setting | What it does | Why |
|---|---|---|
| `random_state=42` | Uses the same "random" shuffle every time | Anyone who reruns the notebook gets the same split, so the same results. This is **reproducibility**. |
| `stratify=real["label"]` | Keeps the same ham/spam mix in both parts | Without it, the test set might get very few spam messages by bad luck |

In [ ]:
# TODO: split `real` into train_df and test_df
# Hint: train_test_split(real, test_size=0.25, random_state=42, stratify=real["label"])
train_df, test_df = None, None

# TODO: print how many rows ended up in each part, and the spam share in each
...

## A3 · Train a model

We use **logistic regression**, one of the simplest classifiers. In plain words, it learns a score like:

> score = (a × length) + (b × punct) + c

turns that score into a **probability of spam** between 0 and 1, and says **spam** when the probability is 0.5 or more. Training means finding the best values of `a`, `b` and `c` from the examples in `train_df`.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    TR["train_df<br/>X and y"] -- "fit" --> M["Trained<br/>model"]
    NEW["New messages<br/>X only"] -- "predict" --> M
    M --> P["Predicted<br/>labels"]
```

In scikit-learn, `X` is the table of features and `y` is the column of labels.

In [ ]:
# TODO: create a LogisticRegression(max_iter=1000) and fit it on the training data
# Hint: model.fit(train_df[FEATURES], train_df[LABEL])
model_v1 = None

In [ ]:
# TODO: take 5 ham and 5 spam rows from test_df,
# and show the model's predictions next to the real labels
# Hint: sample = test_df.groupby("label").head(5)
#       sample.assign(predicted=model_v1.predict(sample[FEATURES]))
...

## A4 · How do we measure a model?

### The confusion matrix

Every prediction falls into one of four boxes. Here **spam is the "positive" class**, the thing we are trying to find.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart TD
    M["A test message"] --> A{"Really spam?"}
    A -- "yes" --> B{"Model said spam?"}
    A -- "no" --> C{"Model said spam?"}
    B -- "yes" --> TP["TP"]
    B -- "no" --> FN["FN"]
    C -- "yes" --> FP["FP"]
    C -- "no" --> TN["TN"]
```

| Box | Name | What happened | Good or bad? |
|---|---|---|---|
| **TP** | True Positive | Spam, and the model caught it | Good |
| **FN** | False Negative | Spam, but it got through to the inbox | Bad |
| **FP** | False Positive | A real message was sent to the spam folder | Bad |
| **TN** | True Negative | A real message was delivered | Good |

A trick to remember the names: the second word is what the model **said** (Positive = spam, Negative = ham). The first word is whether it was **right** (True) or **wrong** (False).

We count each box and put the counts in a 2 × 2 table:

| | predicted ham | predicted spam |
|---|---|---|
| **actual ham** | TN | FP |
| **actual spam** | FN | TP |

### A worked example with 10 messages

Say our test set has 10 messages, 4 of them spam, and the model makes these guesses:

| # | Message | Actual | Predicted | Box |
|---|---|---|---|---|
| 1 | "Lunch at 12?" | ham | ham | TN |
| 2 | "FREE ringtone!!! Text YES" | spam | spam | TP |
| 3 | "Running late, sorry" | ham | ham | TN |
| 4 | "You WON a prize! Call now!" | spam | spam | TP |
| 5 | "Happy birthday!!! 🎉🎉" | ham | **spam** | FP |
| 6 | "Claim your reward today" | spam | **ham** | FN |
| 7 | "Can you call mom?" | ham | ham | TN |
| 8 | "URGENT: account locked!!! Click" | spam | spam | TP |
| 9 | "Meeting moved to 3pm" | ham | ham | TN |
| 10 | "OMG!!! Did you see that?!" | ham | **spam** | FP |

Counting the boxes: **TP = 3, FN = 1, FP = 2, TN = 4**.

| | predicted ham | predicted spam |
|---|---|---|
| **actual ham** | TN = 4 | FP = 2 |
| **actual spam** | FN = 1 | TP = 3 |

### Four metrics from the same four numbers

| Metric | Question it answers | Formula | Our example |
|---|---|---|---|
| **Accuracy** | Of all messages, how many did we get right? | (TP + TN) / all | (3 + 4) / 10 = **70%** |
| **Precision** | When the model says "spam", how often is it right? | TP / (TP + FP) | 3 / (3 + 2) = **60%** |
| **Recall** | Of all the real spam, how much did we catch? | TP / (TP + FN) | 3 / (3 + 1) = **75%** |
| **F1 score** | One number that balances precision and recall | 2 × P × R / (P + R) | 2 × 0.60 × 0.75 / 1.35 = **67%** |

Precision and recall look at different parts of the table:

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart TD
    P0["Precision looks at<br/>what the model<br/>CALLED spam"] --> P1["TP = 3<br/>right"]
    P0 --> P2["FP = 2<br/>wrong"]
    R0["Recall looks at<br/>what REALLY<br/>IS spam"] --> R1["TP = 3<br/>caught"]
    R0 --> R2["FN = 1<br/>missed"]
```

**Which one matters?** It depends on which mistake hurts more:

| Mistake | What the user sees | Metric that punishes it |
|---|---|---|
| False positive (FP) | An important email from your boss lands in spam | **Precision** |
| False negative (FN) | A scam message lands in your inbox | **Recall** |

The **F1 score** is high only when *both* precision and recall are high. It is a harmonic mean, so one bad number drags it down a lot. For example, precision 100% and recall 10% gives F1 = 18%, not 55%.

Section A7 goes much deeper into precision vs recall.

### Try it: compute the metrics by hand

Use the counts from the 10-message example. Then check your answers with scikit-learn in the next cell.

In [ ]:
tp, fn, fp, tn = 3, 1, 2, 4

# TODO: compute each metric from tp, fn, fp, tn using the formulas in the table
accuracy = None
precision = None
recall = None
f1 = None

# print(f"accuracy={accuracy:.2f} precision={precision:.2f} recall={recall:.2f} f1={f1:.2f}")

In [ ]:
# The same 10 messages as lists of labels (run as is to check your answers)
actual    = ["ham", "spam", "ham", "spam", "ham", "spam", "ham", "spam", "ham", "ham"]
predicted = ["ham", "spam", "ham", "spam", "spam", "ham", "ham", "spam", "ham", "spam"]

print("accuracy ", accuracy_score(actual, predicted))
print("precision", precision_score(actual, predicted, pos_label="spam"))
print("recall   ", recall_score(actual, predicted, pos_label="spam"))
print("f1       ", round(f1_score(actual, predicted, pos_label="spam"), 3))

### Now write the evaluate function

`pos_label="spam"` tells scikit-learn that spam is the positive class. `zero_division=0` returns 0 instead of a warning when the model never predicts spam (then precision is 0 / 0).

In [ ]:
def evaluate(model, df):
    y_true = df[LABEL]
    # TODO: predict on df[FEATURES]
    y_pred = None
    return {
        # TODO: accuracy_score(y_true, y_pred)
        "accuracy": None,
        # TODO: precision_score(y_true, y_pred, pos_label="spam", zero_division=0)
        "precision": None,
        # TODO: recall_score(y_true, y_pred, pos_label="spam", zero_division=0)
        "recall": None,
        # TODO: f1_score(y_true, y_pred, pos_label="spam", zero_division=0)
        "f1": None,
        # TODO: confusion_matrix(y_true, y_pred, labels=["ham", "spam"]).tolist()
        "confusion_matrix": None,
    }


def show(metrics):
    """Print the metrics in a readable way."""
    (tn, fp), (fn, tp) = metrics["confusion_matrix"]
    print(f"Accuracy  : {metrics['accuracy']:.1%}   (share of all messages we got right)")
    print(f"Precision : {metrics['precision']:.1%}   (when we say spam, how often we are right)")
    print(f"Recall    : {metrics['recall']:.1%}   (share of real spam we caught)")
    print(f"F1 score  : {metrics['f1']:.1%}   (balance of precision and recall)")
    print()
    print("                predicted ham  predicted spam")
    print(f"actual ham      {tn:>9} TN    {fp:>10} FP")
    print(f"actual spam     {fn:>9} FN    {tp:>10} TP")


# TODO: evaluate model_v1 on test_df and show the result
...

**Q2.** The accuracy looks good. Look at the confusion matrix, precision and recall. What is the model actually doing?

### The accuracy trap

A useful habit: compare your model with the **dumbest possible model**. Here that is a "model" that says ham for every message, without looking at anything.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    ANY["Any message"] --> DUMB["Always-ham<br/>model"] --> H["ham"]
```

If your real model is barely better than this, it has not learned anything useful.

In [ ]:
# TODO: make a list that says "ham" once for every row in test_df
# Hint: ["ham"] * len(test_df)
always_ham = None

# TODO: print the accuracy and recall of always_ham against test_df[LABEL]
...

**Q3.** The always-ham model never catches a single spam. What is its accuracy? Why is accuracy a misleading metric when 85% of messages are ham?

## A5 · Train on the balanced data

One fix for an imbalanced dataset is to train on data with equal numbers of each label. Then the model cannot score well by ignoring spam.

```mermaid
pie showData title balanced.csv, 2,000 messages
    "ham" : 1000
    "spam" : 1000
```

Repeat A1 to A4 with `data/balanced.csv`.

In [ ]:
# TODO: load data/balanced.csv
balanced = None

# TODO: split it the same way as in A2
bal_train, bal_test = None, None

# TODO: train model_v2 on bal_train
model_v2 = None

# TODO: evaluate model_v2 on bal_test and show the result
...

## A6 · Compare both models on the fixed test set

v1 and v2 were each tested on a split of their *own* data. v2's test split is 50% spam, which real life is not. To compare two models fairly, test them on **the same data**, and make that data look like real life.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    V1["model_v1"] --> FT["test_set.csv<br/>same 500 rows<br/>15% spam"]
    V2["model_v2"] --> FT
    FT --> CMP["Compare<br/>side by side"]
```

This is why the pipeline in Part 4 of the lab always uses `data/test_set.csv`.

In [ ]:
# TODO: load data/test_set.csv
fixed_test = None

# TODO: evaluate model_v1 and model_v2 on fixed_test and put the results in a table
# Hint: pd.DataFrame({"v1": {...}, "v2": {...}}).T with accuracy, precision, recall and f1
...

**Q4.** Which model would you put in production? Which metric made the difference clear?

## A7 · Deep dive: precision vs recall

Precision and recall pull in opposite directions. Improving one usually costs you some of the other. This section shows **why**, and **how to choose**.

### A7.1 · The fishing net

Picture a lake with fish and old boots in it. You throw a net.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    LAKE["Lake<br/>fish + boots"] --> NET["Your net"]
    NET --> CATCH["Your catch"]
```

| Fishing | Spam filter |
|---|---|
| a fish | a spam message |
| an old boot | a real (ham) message |
| what ends up in the net | the messages the model calls spam |
| **Recall:** what share of all the fish in the lake did you catch? | Of all real spam, how much did we flag? |
| **Precision:** what share of your catch is fish, not boots? | Of everything we flagged, how much was really spam? |

Now compare two ways of fishing:

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart TD
    BIG["Huge net"] --> B1["Catches every fish"]
    BIG --> B2["Also lots of boots"]
    B1 --> BR["High recall"]
    B2 --> BP["Low precision"]
    SMALL["Small careful net"] --> S1["Only fish"]
    SMALL --> S2["Misses many fish"]
    S1 --> SP["High precision"]
    S2 --> SR["Low recall"]
```

More everyday examples of the same trade-off:

| System | Turned up high (catches everything) | Turned down low (only very sure cases) |
|---|---|---|
| Smoke alarm | Goes off for burnt toast (many false alarms) | Stays quiet for a small real fire |
| Airport security | Searches everyone's bag (slow) | Lets dangerous items through |
| Spam filter | Your boss's email ends up in spam | Scams land in your inbox |

### A7.2 · Where the trade-off comes from: the threshold

The model does not really output "spam" or "ham". It outputs a **probability of spam** between 0 and 1. Then a rule turns that number into a label:

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    MOD["Model"] --> PR["P(spam)<br/>= 0.72"]
    PR --> T{"≥ 0.5?"}
    T -- "yes" --> S["spam"]
    T -- "no" --> H["ham"]
```

Here the **threshold is 0.5**, the default. But nothing forces us to use 0.5. Moving it changes which mistakes we make:

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    LOW["Lower threshold<br/>e.g. 0.3"] --> L1["Flags MORE<br/>messages"]
    L1 --> L2["Recall up<br/>Precision down"]
    HIGH["Higher threshold<br/>e.g. 0.8"] --> H1["Flags FEWER<br/>messages"]
    H1 --> H2["Precision up<br/>Recall down"]
```

**A worked example.** A model gave these 10 messages a probability of spam, sorted from most to least spammy. 4 of them are really spam.

| # | Message | Actual | P(spam) |
|---|---|---|---|
| 1 | "WIN cash now!!! Text WIN to 80082" | spam | 0.95 |
| 2 | "FREE entry to our prize draw!!!" | spam | 0.85 |
| 3 | "Party tonight!!! Bring snacks!!!" | ham | 0.75 |
| 4 | "Your package is waiting, click here" | spam | 0.65 |
| 5 | "Did you get my long email about the trip?" | ham | 0.55 |
| 6 | "Hi, is this still your number?" | spam | 0.45 |
| 7 | "Call me when you land" | ham | 0.35 |
| 8 | "Thanks for dinner :)" | ham | 0.25 |
| 9 | "ok" | ham | 0.15 |
| 10 | "See you tomorrow" | ham | 0.05 |

Everything **at or above** the threshold is flagged as spam. Watch what happens at three thresholds:

| Threshold | Flagged as spam | TP | FP | FN | Precision | Recall |
|---|---|---|---|---|---|---|
| 0.8 | #1, #2 | 2 | 0 | 2 | 2 / 2 = **100%** | 2 / 4 = **50%** |
| 0.5 | #1 to #5 | 3 | 2 | 1 | 3 / 5 = **60%** | 3 / 4 = **75%** |
| 0.3 | #1 to #7 | 4 | 3 | 0 | 4 / 7 = **57%** | 4 / 4 = **100%** |

- At **0.8** the model only flags the two most obvious messages. Both are right (precision 100%), but it misses half the spam.
- At **0.3** it catches all four spam messages (recall 100%), but three of its seven flags are real messages: the party invite, the long email, and "Call me when you land".
- Message #6, "Hi, is this still your number?", is a sneaky spam that looks like a normal text. The only way to catch it is to lower the threshold, and that drags in real messages too.

**Your turn:** work out the numbers for **threshold = 0.6**.

In [ ]:
# 10 messages with the probability of spam a model gave each one (made-up numbers)
toy = pd.DataFrame({
    "message": [
        "WIN cash now!!! Text WIN to 80082",
        "FREE entry to our prize draw!!!",
        "Party tonight!!! Bring snacks!!!",
        "Your package is waiting, click here",
        "Did you get my long email about the trip?",
        "Hi, is this still your number?",
        "Call me when you land",
        "Thanks for dinner :)",
        "ok",
        "See you tomorrow",
    ],
    "actual": ["spam", "spam", "ham", "spam", "ham", "spam", "ham", "ham", "ham", "ham"],
    "p_spam": [0.95, 0.85, 0.75, 0.65, 0.55, 0.45, 0.35, 0.25, 0.15, 0.05],
})
toy

In [ ]:
threshold = 0.6

# TODO: label each message "spam" if p_spam >= threshold, else "ham"
# Hint: np.where(toy["p_spam"] >= threshold, "spam", "ham")
pred = None

# TODO: count TP, FP and FN
# Hint: tp = ((pred == "spam") & (toy["actual"] == "spam")).sum()
tp = None
fp = None
fn = None

# TODO: compute precision and recall from the counts
precision = None
recall = None

# print(f"threshold={threshold}  TP={tp} FP={fp} FN={fn}  precision={precision:.0%}  recall={recall:.0%}")

**Q5.** At threshold 0.6, which messages are flagged? Compared with 0.5, which mistake did you get rid of, and what did it cost you?

### A7.3 · The two cheats

Either metric on its own is easy to "win" with a useless model:

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart TD
    A["Always say spam"] --> A1["Recall 100%<br/>no spam missed"]
    A --> A2["Precision low<br/>every ham flagged"]
    N["Never say spam"] --> N1["Precision 100%?<br/>no wrong flags"]
    N --> N2["Recall 0%<br/>all spam missed"]
```

That is why we **always report precision and recall together**, or use F1, which is only high when both are high.

In [ ]:
# TODO: an "always spam" model: a list with "spam" for every row in test_df
always_spam = None

# TODO: print its precision, recall and F1 against test_df[LABEL]
# Hint: precision_score(test_df[LABEL], always_spam, pos_label="spam", zero_division=0)
...

**Q6.** Compare the always-spam precision with the share of spam in `test_df`. Why are they the same number?

### A7.4 · Sweep the threshold on a real model

Now do the same thing as in A7.2, but with `model_v1` on all 250 messages in `test_df`. `predict_proba` gives the probability of each label. We pick the spam column.

In [ ]:
def bar(x, width=20):
    """A tiny text bar chart: bar(0.5) -> 10 blocks."""
    return "█" * round(x * width)

In [ ]:
# TODO: get the probability of spam for each message in test_df
# Hint: spam_col = list(model_v1.classes_).index("spam")
#       proba = model_v1.predict_proba(test_df[FEATURES])[:, spam_col]
proba = None

rows = []
for threshold in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]:
    # TODO: say "spam" where proba >= threshold, else "ham"
    pred = None
    # TODO: append a dict with threshold, TP, FP, FN, precision, recall and f1
    # Hint: rows.append({"threshold": threshold, "TP": ..., "FP": ..., ...})
    ...

sweep = pd.DataFrame(rows)
sweep

In [ ]:
# Run as is once the sweep above works: a text chart of the trade-off
for _, r in sweep.iterrows():
    print(f"{r['threshold']:.2f}  precision {bar(r['precision']):<20} {r['precision']:.2f}"
          f"   recall {bar(r['recall']):<20} {r['recall']:.2f}")

**Q7.** Which threshold gives the best F1? Something strange happens between 0.3 and 0.4. Look at the TP and FP columns: what does it tell you about how confident `model_v1` is?

### A7.5 · Choose the threshold by what mistakes cost

F1 treats both mistakes as equally bad. In real life they rarely are. For a spam filter, say:

- a **false positive** (a real message hidden in spam) costs **10** points: you might miss a job offer
- a **false negative** (a spam in your inbox) costs **1** point: you just delete it

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    FP["Each FP"] -- "× 10" --> C["Total cost"]
    FN["Each FN"] -- "× 1" --> C
    C --> PICK["Pick the threshold<br/>with lowest cost"]
```

> total cost = 10 × FP + 1 × FN

Use `model_v2` (the good model) on `fixed_test` this time.

In [ ]:
COST_FP = 10   # a real message lost in the spam folder
COST_FN = 1    # a spam message in the inbox

# TODO: get the probability of spam from model_v2 for every row in fixed_test
proba_v2 = None

rows = []
for threshold in [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]:
    # TODO: label, count FP and FN, and compute the cost
    pred = None
    fp = None
    fn = None
    # rows.append({"threshold": threshold, "FP": fp, "FN": fn, "cost": COST_FP * fp + COST_FN * fn})
    ...

pd.DataFrame(rows)

**Q8.** Which threshold has the lowest cost? Now imagine a **fraud detector**: a missed fraud (FN) costs 50, and a false alarm (FP) costs 1 because a person just checks it. Change `COST_FP` and `COST_FN` and run again. Does the best threshold go up or down?

> **Careful:** we picked the threshold by looking at the test set. In a real project you would pick it on a separate **validation set**, and keep the test set for the final check only. Otherwise the test score is a little too optimistic.

### A7.6 · Which metric when?

| Situation | The worse mistake | Focus on | Why |
|---|---|---|---|
| Spam filter | FP: a real email is hidden | **Precision** | People forgive some spam, not a lost email |
| Cancer screening | FN: a sick patient is sent home | **Recall** | A false alarm only means one more test |
| Fraud alerts | FN: fraud goes through | **Recall** | A person reviews each alert anyway |
| Search results, page 1 | FP: junk at the top | **Precision** | Users only look at the first few results |
| Kids video filter | FN: an unsafe video is shown | **Recall** | Blocking a few safe videos is fine |
| Balanced, no strong preference | both | **F1** | One number that needs both to be good |

> **Going further:** the **F-beta score** lets you choose the balance. F2 counts recall twice as much as precision, F0.5 counts precision twice as much. In scikit-learn: `fbeta_score(y_true, y_pred, beta=2, pos_label="spam")`.

## A8 · Save the model and its model card

A `.pkl` file on its own does not say how the model was made. Six months from now nobody will remember. So we save a **model card** next to it.

We save **both** models, v1 and v2. The next notebook, `mlops_pipeline.ipynb`, puts them into a model registry and runs them through an MLOps pipeline.

```mermaid
%%{init: {"htmlLabels": false, "flowchart": {"htmlLabels": false}}}%%
flowchart LR
    M["model_v2"] -- "joblib.dump" --> PKL["model_nb_v2.pkl<br/>the model"]
    M -- "json.dump" --> CARD["model_nb_v2.json<br/>the model card"]
    CARD --- W1["created_at"]
    CARD --- W2["data file +<br/>fingerprint"]
    CARD --- W3["algorithm"]
    CARD --- W4["metrics"]
```

In [ ]:
def file_fingerprint(path):
    """Short SHA-256 hash of a file. Same data -> same fingerprint."""
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()[:12]


def save_with_card(model, version, data_file):
    """Save a model to models/model_<version>.pkl and its card to models/model_<version>.json."""
    os.makedirs("models", exist_ok=True)

    # TODO: save the model to models/model_<version>.pkl
    # Hint: joblib.dump(model, f"models/model_{version}.pkl")
    ...

    # TODO: fill in the model card
    card = {
        "version": version,
        "created_at": None,        # TODO: datetime.now().isoformat(timespec="seconds")
        "data_file": data_file,
        "data_fingerprint": None,  # TODO: file_fingerprint(data_file)
        "rows": len(pd.read_csv(data_file)),
        "algorithm": "LogisticRegression",
        "metrics": None,           # TODO: evaluate(model, fixed_test)
    }

    # TODO: write the card to models/model_<version>.json
    # Hint: json.dump(card, f, indent=2)
    ...
    return card


# Save BOTH models. The MLOps notebook (mlops_pipeline.ipynb) picks them up from here.
card_v1 = save_with_card(model_v1, "nb_v1", "data/sms_real_1k.csv")
card_v2 = save_with_card(model_v2, "nb_v2", "data/balanced.csv")
print(json.dumps(card_v2, indent=2))

**Q9.** What is `data_fingerprint`? When would it change, and why is that useful six months from now?

## A9 · Use the saved model

This is what a web app or API does: load the saved file, turn new text into features, and predict.

```mermaid
sequenceDiagram
    participant U as User
    participant A as App
    participant M as Saved model
    U->>A: message text
    A->>A: extract_features
    A->>M: predict(length, punct)
    M-->>A: ham or spam
    A-->>U: show the answer
```

In [ ]:
def extract_features(text):
    """Turn a raw SMS message into the two numbers our model understands."""
    return {
        "length": len(text),
        "punct": sum(1 for ch in text if ch in string.punctuation),
    }


# TODO: load models/model_nb_v2.pkl
loaded = None

for message in [
    "See you at lunch",
    "URGENT! You have won a 1000 dollar gift card. Call 0800-123-456 now to claim your prize, offer ends today!!! Reply STOP to opt out.",
    "WINNER!! Claim your free prize now!!!",
]:
    # TODO: turn the message into a one-row DataFrame and predict
    # Hint: loaded.predict(pd.DataFrame([extract_features(message)]))[0]
    prediction = None
    print(f"{prediction!s:>5}  <- {message[:60]}")

**Q10.** The last message is obviously spam. Why does the model say ham? What would a better model need to look at?

## Cheat sheet

| Metric | Formula | High means | Use it when |
|---|---|---|---|
| Accuracy | (TP + TN) / all | Most predictions are right | Labels are roughly balanced |
| Precision | TP / (TP + FP) | Few false alarms | False alarms are costly (good email lost) |
| Recall | TP / (TP + FN) | Few misses | Misses are costly (scam gets through, disease missed) |
| F1 score | 2PR / (P + R) | Both precision and recall are good | You need one number and labels are imbalanced |

| Threshold | Effect |
|---|---|
| Lower it | More flagged as spam: recall up, precision down |
| Raise it | Fewer flagged as spam: precision up, recall down |
| Choose it | By the cost of each mistake, on a validation set |

---
# Part B · Solution

Try Part A first. This section runs on its own from top to bottom. The explanations are in Part A; here you get the code and the answers.

## B0 · Setup

In [ ]:
import hashlib
import json
import os
import string
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split

FEATURES = ["length", "punct"]
LABEL = "label"

for f in ["data/sms_real_1k.csv", "data/balanced.csv", "data/test_set.csv"]:
    assert os.path.exists(f), f"{f} is missing. Run `python make_data.py` in this notebook's folder first."
print("Data files found.")

## B1 · Load and explore the real data

In [ ]:
real = pd.read_csv("data/sms_real_1k.csv")
real.head()

In [ ]:
print(real["label"].value_counts(normalize=True).round(2))
real.groupby("label")[FEATURES].mean().round(1)

**A1.** 15% of messages are spam. On average spam is about twice as long and has a little more punctuation, but many spam messages still look like ham on these two numbers.

## B2 · Split into train and test

In [ ]:
train_df, test_df = train_test_split(real, test_size=0.25, random_state=42, stratify=real["label"])
print(f"train: {len(train_df)} rows, spam share {(train_df[LABEL] == 'spam').mean():.0%}")
print(f"test : {len(test_df)} rows, spam share {(test_df[LABEL] == 'spam').mean():.0%}")

## B3 · Train a model

In [ ]:
model_v1 = LogisticRegression(max_iter=1000)
model_v1.fit(train_df[FEATURES], train_df[LABEL])

In [ ]:
sample = test_df.groupby("label").head(5)
sample.assign(predicted=model_v1.predict(sample[FEATURES]))

## B4 · Metrics by hand

In [ ]:
tp, fn, fp, tn = 3, 1, 2, 4

accuracy = (tp + tn) / (tp + fn + fp + tn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

print(f"accuracy={accuracy:.2f} precision={precision:.2f} recall={recall:.2f} f1={f1:.2f}")

In [ ]:
actual    = ["ham", "spam", "ham", "spam", "ham", "spam", "ham", "spam", "ham", "ham"]
predicted = ["ham", "spam", "ham", "spam", "spam", "ham", "ham", "spam", "ham", "spam"]

print("accuracy ", accuracy_score(actual, predicted))
print("precision", precision_score(actual, predicted, pos_label="spam"))
print("recall   ", recall_score(actual, predicted, pos_label="spam"))
print("f1       ", round(f1_score(actual, predicted, pos_label="spam"), 3))

## B4 · Evaluate model_v1

In [ ]:
def evaluate(model, df):
    y_true = df[LABEL]
    y_pred = model.predict(df[FEATURES])
    return {
        "accuracy": round(accuracy_score(y_true, y_pred), 3),
        "precision": round(precision_score(y_true, y_pred, pos_label="spam", zero_division=0), 3),
        "recall": round(recall_score(y_true, y_pred, pos_label="spam", zero_division=0), 3),
        "f1": round(f1_score(y_true, y_pred, pos_label="spam", zero_division=0), 3),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=["ham", "spam"]).tolist(),
    }


def show(metrics):
    """Print the metrics in a readable way."""
    (tn, fp), (fn, tp) = metrics["confusion_matrix"]
    print(f"Accuracy  : {metrics['accuracy']:.1%}   (share of all messages we got right)")
    print(f"Precision : {metrics['precision']:.1%}   (when we say spam, how often we are right)")
    print(f"Recall    : {metrics['recall']:.1%}   (share of real spam we caught)")
    print(f"F1 score  : {metrics['f1']:.1%}   (balance of precision and recall)")
    print()
    print("                predicted ham  predicted spam")
    print(f"actual ham      {tn:>9} TN    {fp:>10} FP")
    print(f"actual spam     {fn:>9} FN    {tp:>10} TP")


show(evaluate(model_v1, test_df))

**A2.** The model predicts ham for almost every message. It catches none of the 38 spam messages (recall 0%), and the 3 messages it does call spam are all wrong (precision 0%), so F1 is 0%. Its 83.6% accuracy comes almost entirely from the fact that most messages are ham.

## B4 · The accuracy trap

In [ ]:
always_ham = ["ham"] * len(test_df)

print(f"accuracy: {accuracy_score(test_df[LABEL], always_ham):.1%}")
print(f"recall  : {recall_score(test_df[LABEL], always_ham, pos_label='spam', zero_division=0):.1%}")

**A3.** The always-ham model gets 84.8% accuracy while catching zero spam. That is *higher* than model_v1's 83.6%. When one label is 85% of the data, you get about 85% accuracy for free by always guessing it. Accuracy hides the fact that the model is useless at the job we care about. Recall and F1 show it straight away.

## B5 · Train on the balanced data

In [ ]:
balanced = pd.read_csv("data/balanced.csv")
bal_train, bal_test = train_test_split(balanced, test_size=0.25, random_state=42, stratify=balanced["label"])

model_v2 = LogisticRegression(max_iter=1000)
model_v2.fit(bal_train[FEATURES], bal_train[LABEL])

show(evaluate(model_v2, bal_test))

## B6 · Compare both models on the fixed test set

In [ ]:
fixed_test = pd.read_csv("data/test_set.csv")

keep = ["accuracy", "precision", "recall", "f1"]
m1 = evaluate(model_v1, fixed_test)
m2 = evaluate(model_v2, fixed_test)
pd.DataFrame({
    "v1 (real, imbalanced)": {k: m1[k] for k in keep},
    "v2 (balanced)": {k: m2[k] for k in keep},
}).T

**A4.** v2. Both models have high accuracy (86% vs 99%), so accuracy alone makes them look close. Recall (4% vs 100%) and F1 (8% vs 97%) show the real gap: v1 lets almost all spam through.

Notice v1's precision is 100%. That sounds perfect, but it only flagged 3 messages as spam (and caught 3 of 75). Precision alone can look great for a model that hardly ever says spam. That is why we look at precision **and** recall, or at F1.

## B7.2 · Threshold 0.6 on the 10 messages

In [ ]:
# 10 messages with the probability of spam a model gave each one (made-up numbers)
toy = pd.DataFrame({
    "message": [
        "WIN cash now!!! Text WIN to 80082",
        "FREE entry to our prize draw!!!",
        "Party tonight!!! Bring snacks!!!",
        "Your package is waiting, click here",
        "Did you get my long email about the trip?",
        "Hi, is this still your number?",
        "Call me when you land",
        "Thanks for dinner :)",
        "ok",
        "See you tomorrow",
    ],
    "actual": ["spam", "spam", "ham", "spam", "ham", "spam", "ham", "ham", "ham", "ham"],
    "p_spam": [0.95, 0.85, 0.75, 0.65, 0.55, 0.45, 0.35, 0.25, 0.15, 0.05],
})
toy

In [ ]:
threshold = 0.6

pred = np.where(toy["p_spam"] >= threshold, "spam", "ham")

tp = ((pred == "spam") & (toy["actual"] == "spam")).sum()
fp = ((pred == "spam") & (toy["actual"] == "ham")).sum()
fn = ((pred == "ham") & (toy["actual"] == "spam")).sum()

precision = tp / (tp + fp)
recall = tp / (tp + fn)

print(f"threshold={threshold}  TP={tp} FP={fp} FN={fn}  precision={precision:.0%}  recall={recall:.0%}")
toy.assign(predicted=pred)

**A5.** At 0.6 the model flags #1 to #4. Compared with 0.5, message #5 ("Did you get my long email about the trip?", p = 0.55) is no longer flagged, so one false positive is gone and precision rises from 60% to 75%. Recall stays at 75% because #5 was ham anyway. Here raising the threshold was a free win. Raise it once more, past 0.65, and you would lose the real spam #4, and recall would drop.

## B7.3 · The two cheats

In [ ]:
always_spam = ["spam"] * len(test_df)

print(f"precision: {precision_score(test_df[LABEL], always_spam, pos_label='spam'):.1%}")
print(f"recall   : {recall_score(test_df[LABEL], always_spam, pos_label='spam'):.1%}")
print(f"f1       : {f1_score(test_df[LABEL], always_spam, pos_label='spam'):.1%}")
print(f"spam share of test_df: {(test_df[LABEL] == 'spam').mean():.1%}")

**A6.** Precision is 15.2%, exactly the spam share of `test_df`. When you flag everything, precision = TP / (TP + FP) = all spam / all messages, which *is* the spam share. Recall is a "perfect" 100%, yet the model is useless. F1 (26%) gives it away.

## B7.4 · Sweep the threshold on model_v1

In [ ]:
def bar(x, width=20):
    """A tiny text bar chart: bar(0.5) -> 10 blocks."""
    return "█" * round(x * width)

In [ ]:
spam_col = list(model_v1.classes_).index("spam")
proba = model_v1.predict_proba(test_df[FEATURES])[:, spam_col]

rows = []
for threshold in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]:
    pred = np.where(proba >= threshold, "spam", "ham")
    (tn, fp), (fn, tp) = confusion_matrix(test_df[LABEL], pred, labels=["ham", "spam"])
    rows.append({
        "threshold": threshold,
        "TP": tp, "FP": fp, "FN": fn,
        "precision": round(precision_score(test_df[LABEL], pred, pos_label="spam", zero_division=0), 3),
        "recall": round(recall_score(test_df[LABEL], pred, pos_label="spam", zero_division=0), 3),
        "f1": round(f1_score(test_df[LABEL], pred, pos_label="spam", zero_division=0), 3),
    })

sweep = pd.DataFrame(rows)
sweep

In [ ]:
for _, r in sweep.iterrows():
    print(f"{r['threshold']:.2f}  precision {bar(r['precision']):<20} {r['precision']:.2f}"
          f"   recall {bar(r['recall']):<20} {r['recall']:.2f}")

**A7.** The best F1 is about 0.64 at threshold 0.25. That is far better than the 0 we got at the default 0.5, without retraining anything.

Between 0.3 and 0.4, TP drops from 24 to 2. So 22 of the 38 spam messages got a probability between 0.3 and 0.4, and only 2 got more than 0.4: `model_v1` is almost never confident that anything is spam. It learned from data that was 85% ham, so it keeps all its guesses low. At the default threshold of 0.5 it flags almost nothing. Moving the threshold helps a weak model, but it is a patch. Better data (model_v2) is the real fix.

## B7.5 · Choose the threshold by cost

In [ ]:
COST_FP = 10   # a real message lost in the spam folder
COST_FN = 1    # a spam message in the inbox

spam_col = list(model_v2.classes_).index("spam")
proba_v2 = model_v2.predict_proba(fixed_test[FEATURES])[:, spam_col]

rows = []
for threshold in [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]:
    pred = np.where(proba_v2 >= threshold, "spam", "ham")
    fp = ((pred == "spam") & (fixed_test[LABEL] == "ham")).sum()
    fn = ((pred == "ham") & (fixed_test[LABEL] == "spam")).sum()
    rows.append({"threshold": threshold, "FP": fp, "FN": fn, "cost": COST_FP * fp + COST_FN * fn})

pd.DataFrame(rows)

**A8.** With FP = 10 and FN = 1, the lowest cost is at threshold **0.9** (cost 0 on this test set). The default 0.5 costs 50, because of 5 real messages sent to spam. Going even higher, to 0.99, starts letting spam through again.

For the fraud detector (FP = 1, FN = 50), misses become expensive, so the best threshold moves **down**: you flag more, accept more false alarms, and let a person check them.

## B8 · Save the model and its model card

In [ ]:
def file_fingerprint(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()[:12]


def save_with_card(model, version, data_file):
    os.makedirs("models", exist_ok=True)
    joblib.dump(model, f"models/model_{version}.pkl")
    card = {
        "version": version,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "data_file": data_file,
        "data_fingerprint": file_fingerprint(data_file),
        "rows": len(pd.read_csv(data_file)),
        "algorithm": "LogisticRegression",
        "metrics": evaluate(model, fixed_test),
    }
    with open(f"models/model_{version}.json", "w") as f:
        json.dump(card, f, indent=2)
    return card


card_v1 = save_with_card(model_v1, "nb_v1", "data/sms_real_1k.csv")
card_v2 = save_with_card(model_v2, "nb_v2", "data/balanced.csv")
print(json.dumps(card_v2, indent=2))

**A9.** It is a short hash of the training data file. If even one row changes, the fingerprint changes. Months later, when the model misbehaves, you can check exactly which data it was trained on and whether the data you have now is the same.

## B9 · Use the saved model

In [ ]:
def extract_features(text):
    return {
        "length": len(text),
        "punct": sum(1 for ch in text if ch in string.punctuation),
    }


loaded = joblib.load("models/model_nb_v2.pkl")

for message in [
    "See you at lunch",
    "URGENT! You have won a 1000 dollar gift card. Call 0800-123-456 now to claim your prize, offer ends today!!! Reply STOP to opt out.",
    "WINNER!! Claim your free prize now!!!",
]:
    features = extract_features(message)
    prediction = loaded.predict(pd.DataFrame([features]))[0]
    print(f"{prediction:>5}  {features}  <- {message[:60]}")

**A10.** The message is short (37 characters), and in our training data short messages are almost always ham. The model never sees the words. A better model would use the text itself: words like "WINNER", "free" or "prize" (for example with TF-IDF features), or a deep learning model that learns meaning from the text.